# IoT MCP Tools — a guided tour

AssetOpsBench ships an **MCP server per industrial domain**. This notebook walks
through the **IoT server** end to end: every one of its 12 tools, in the order you
would actually reach for them, using the `ToolUniverse` client from `mcphub`.

By the end you will have gone from *"I know nothing about this plant"* to a plotted
sensor trace and a reusable multi-tool workflow, without writing a line of MCP
protocol code.

**What you need before starting** (see [`docs/data.md`](../docs/data.md)):

```bash
uv sync                                                    # install dependencies
docker compose -f src/couchdb/docker-compose.yaml up -d    # start CouchDB
cp .env.public .env                                        # COUCHDB_* connection settings
```

Verify the data layer is up before running anything below:

```bash
curl -s -u admin:password http://localhost:5984/_all_dbs
# → ["workorder","iot","asset","vibration", ...]
```

If `iot` or `asset` is missing, load the default dataset from the host:

```bash
uv run python src/couchdb/init_data.py
```

> **Note:** this notebook talks to a CouchDB running on `localhost`, so it is meant
> to be run locally (`uv run jupyter lab`). It will not work on hosted Colab
> without first exposing a reachable CouchDB and pointing `COUCHDB_URL` at it.

## 1. Connect

`ToolUniverse` follows a three-step contract: **init → load → run**. `load_tools()`
launches each MCP server as a subprocess and discovers the tools it advertises, so
the first call takes a moment.

Tools are namespaced `<server>.<tool>`, e.g. `iot.sites`.

In [ ]:
from mcphub import ToolUniverse

tu = ToolUniverse()                 # 1. init
n = tu.load_tools(servers=["iot"])  # 2. load (connect + discover)
print(f"loaded {n} tools")

Everything below uses two small helpers: `show()` to pretty-print a tool result, and
`unwrap()` because most IoT tools nest their payload under a `result` key.

In [ ]:
import json


def show(title, obj, limit=None):
    print(f"=== {title} ===")
    text = json.dumps(obj, indent=2, default=str)
    if limit and len(text) > limit:
        text = text[:limit] + "\n... (truncated)"
    print(text)


def unwrap(res):
    """Most IoT tools wrap their payload in {'result': ...}; a few do not."""
    return res.get("result", res) if isinstance(res, dict) else res

## 2. Discover what is available

Before calling anything, ask the server what it can do. Three discovery calls matter:

| Call | Answers |
|---|---|
| `tu.list_tools("iot")` | what tools exist |
| `tu.find_tools("...")` | which tool matches an intent |
| `tu.tool_specification(name)` | what arguments a tool takes |

In [ ]:
tools = tu.list_tools("iot")
print(f"{len(tools)} IoT tools:")
for t in tools:
    print("  -", t)

In [ ]:
# Keyword search across loaded tools — useful when you know the intent, not the name.
show("find 'sensor history'", [t["name"] for t in tu.find_tools("sensor history")])

In [ ]:
# The full input schema for one tool, straight from the server.
show("spec: iot.history", tu.tool_specification("iot.history"))

## 3. Where is everything? (sites and assets)

Start at the top of the hierarchy: **site → asset → sensor**.

- `iot.sites` — every site in the registry
- `iot.asset_ids` — asset IDs at one site (just the names)
- `iot.assets` — the same assets with detail, optionally filtered by type
- `iot.asset_detail` — the full record for one asset

In [ ]:
sites = unwrap(tu.run({"name": "iot.sites", "arguments": {}}))
show("iot.sites", sites)

SITE = sites["sites"][0]   # everything below uses the first site
print("\nusing site:", SITE)

In [ ]:
show("iot.asset_ids", unwrap(tu.run({
    "name": "iot.asset_ids",
    "arguments": {"site_name": SITE},
})))

In [ ]:
# `assets` returns records rather than bare IDs, and takes an optional type filter.
show("iot.assets (all)", unwrap(tu.run({
    "name": "iot.assets",
    "arguments": {"site_name": SITE},
})))

In [ ]:
show("iot.assets (assettype=CHILLER)", unwrap(tu.run({
    "name": "iot.assets",
    "arguments": {"site_name": SITE, "assettype": "CHILLER"},
})))

In [ ]:
# Pick the asset with the most sensors — it makes the rest of the tour more interesting.
all_assets = unwrap(tu.run({"name": "iot.assets", "arguments": {"site_name": SITE}}))["assets"]
ASSET = max(all_assets, key=lambda a: a.get("n_sensors", 0))["asset_id"]
print("using asset:", ASSET)

show("iot.asset_detail", unwrap(tu.run({
    "name": "iot.asset_detail",
    "arguments": {"site_name": SITE, "asset_id": ASSET},
})))

## 4. What is measured? (sensor inventory)

Two different questions that are easy to confuse:

- `iot.installed_sensors` — what the **registry says** is fitted to the asset
- `iot.measured_sensors` — what actually **appears in the telemetry**

They can disagree, and that gap is often the interesting part: a sensor that is
installed but never reports is a data-collection problem, not a healthy asset.

In [ ]:
def sensor_sets(site_name, asset_id):
    installed = unwrap(tu.run({
        "name": "iot.installed_sensors",
        "arguments": {"site_name": site_name, "asset_id": asset_id},
    })).get("sensors", [])
    measured = unwrap(tu.run({
        "name": "iot.measured_sensors",
        "arguments": {"site_name": site_name, "asset_id": asset_id},
    })).get("sensors", [])
    return set(installed), set(measured)


# Sweep the whole site rather than one asset — the disagreements are the point.
print(f"{'asset':<12} {'installed':>9} {'measured':>9} {'only installed':>15} {'only measured':>14}")
for a in all_assets:
    inst, meas = sensor_sets(SITE, a["asset_id"])
    print(f"{a['asset_id']:<12} {len(inst):>9} {len(meas):>9} "
          f"{len(inst - meas):>15} {len(meas - inst):>14}")

Two different failure shapes show up here:

- an asset where a sensor is **installed but never reports** — a data-collection gap
- an asset where the counts match but the **names** do not, so a sensor is present
  in the registry under one name and in the telemetry under another

Both matter before you trust an analysis. Drill into whichever asset shows a
non-zero column:

In [ ]:
for a in all_assets:
    inst, meas = sensor_sets(SITE, a["asset_id"])
    if inst - meas or meas - inst:
        print(f"--- {a['asset_id']}")
        for s in sorted(inst - meas):
            print("    registry only :", s)
        for s in sorted(meas - inst):
            print("    telemetry only:", s)

`iot.sensor_coverage` answers the same question quantitatively: per sensor, how many
non-null readings exist and over what period.

In [ ]:
coverage = unwrap(tu.run({
    "name": "iot.sensor_coverage",
    "arguments": {"site_name": SITE, "asset_id": ASSET},
}))
print(f"docs scanned: {coverage['docs_scanned']}\n")
for s in coverage["sensors"][:5]:
    print(f"  {s['sensor'][:55]:<55} {s['non_null_count']:>6} readings")
print(f"  ... {len(coverage['sensors'])} sensors total")

### Searching the other way round: sensors → assets

`iot.find_assets_by_sensors` inverts the lookup. Use it when you know the measurement
you care about but not which assets provide it.

- `match="all"` (default) requires every listed sensor; `match="any"` requires one
- `substring=True` matches on a fragment, which matters here because sensor names are
  prefixed with the asset name
- `source="measured"` (default) searches telemetry; `"installed"` searches the registry

In [ ]:
show("find_assets_by_sensors: anything with 'Temperature'", unwrap(tu.run({
    "name": "iot.find_assets_by_sensors",
    "arguments": {
        "site_name": SITE,
        "sensors": ["Temperature"],
        "match": "any",
        "substring": True,
    },
})))

## 5. The data itself

Four tools, from cheapest to most detailed:

| Tool | Use it to |
|---|---|
| `iot.stream_extent` | check the time range and record count *before* pulling data |
| `iot.latest_reading` | get the most recent value for every sensor |
| `iot.sensor_stats` | min / max / mean / stddev per sensor |
| `iot.history` | the raw observations, paginated |

Always call `stream_extent` first. It tells you whether a `history` call is going to
exceed the page limit, which saves you from an accidental full-stream pull.

In [ ]:
extent = unwrap(tu.run({
    "name": "iot.stream_extent",
    "arguments": {"site_name": SITE, "asset_id": ASSET},
}))
show("iot.stream_extent", extent)

In [ ]:
latest = unwrap(tu.run({
    "name": "iot.latest_reading",
    "arguments": {"site_name": SITE, "asset_id": ASSET},
}))
print("as of", latest["timestamp"], "\n")
for name, value in list(latest["values"].items())[:6]:
    print(f"  {name[:55]:<55} {value}")

In [ ]:
stats = unwrap(tu.run({
    "name": "iot.sensor_stats",
    "arguments": {"site_name": SITE, "asset_id": ASSET},
}))
for s in stats["stats"][:5]:
    print(f"{s['sensor'][:45]:<45} n={s['count']:<6} "
          f"min={s['min']:>10.2f} max={s['max']:>10.2f} mean={s['mean']:>10.2f}")

### Pulling history

`iot.history` is paginated. Pass `limit` to bound a page and feed the returned
`cursor` back in to continue. Narrow the pull with `sensors=[...]` and a
`start`/`end` window rather than fetching everything and filtering client-side.

In [ ]:
SENSOR = stats["stats"][0]["sensor"]
print("focus sensor:", SENSOR)

page = unwrap(tu.run({
    "name": "iot.history",
    "arguments": {
        "site_name": SITE,
        "asset_id": ASSET,
        "sensors": [SENSOR],
        "limit": 5,
    },
}))
show("iot.history (first page)", page, limit=1200)

In [ ]:
# The page tells you whether more exists (`has_more`) and how to ask for it
# (`next_cursor`). Feed that cursor back in unchanged; it is opaque and bound
# to this exact query.
print("returned :", page["returned"])
print("has_more :", page["has_more"])

if page["has_more"]:
    page2 = unwrap(tu.run({
        "name": "iot.history",
        "arguments": {
            "site_name": SITE,
            "asset_id": ASSET,
            "sensors": [SENSOR],
            "limit": 5,
            "cursor": page["next_cursor"],
        },
    }))
    print("\npage 1 timestamps:", [o["timestamp"] for o in page["observations"]])
    print("page 2 timestamps:", [o["timestamp"] for o in page2["observations"]])
else:
    print("the whole stream fitted in one page")

## 6. Plot it

Nothing MCP-specific here: pull a window of history and hand the observations to
pandas and matplotlib.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    %pip install -q matplotlib
    import matplotlib.pyplot as plt

import pandas as pd

In [ ]:
window = unwrap(tu.run({
    "name": "iot.history",
    "arguments": {
        "site_name": SITE,
        "asset_id": ASSET,
        "sensors": [SENSOR],
        "start": extent["start_time"],
        "end": extent["end_time"],
        "limit": 500,
    },
}))

df = pd.DataFrame(window["observations"])
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.set_index("timestamp").sort_index()
print(df.shape)
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
df[SENSOR].plot(ax=ax, linewidth=0.9)
ax.set_title(f"{ASSET} — {SENSOR}")
ax.set_xlabel("")
ax.set_ylabel(SENSOR.replace(f"{ASSET} ", ""))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Chaining tools into a workflow

Calling tools one at a time is fine interactively, but a real question ("give me a
health snapshot of this asset") is several calls stitched together. MCPHub lets you
register that as a **workflow** and then invoke it through the *same* `tu.run()`
entrypoint as any single tool.

A workflow is just a function `fn(tu, **arguments)`. Register it at runtime with
`tu.register_workflow(...)`, or add it permanently by putting it in
`src/mcphub/workflows.py` and listing its name in `REGISTERED`.

In [ ]:
def asset_snapshot(tu, site_name, asset_id, top_n=5):
    """Registry detail + data extent + coverage gap, in one call."""
    detail = unwrap(tu.run({
        "name": "iot.asset_detail",
        "arguments": {"site_name": site_name, "asset_id": asset_id},
    }))
    extent = unwrap(tu.run({
        "name": "iot.stream_extent",
        "arguments": {"site_name": site_name, "asset_id": asset_id},
    }))
    installed = unwrap(tu.run({
        "name": "iot.installed_sensors",
        "arguments": {"site_name": site_name, "asset_id": asset_id},
    }))["sensors"]
    measured = unwrap(tu.run({
        "name": "iot.measured_sensors",
        "arguments": {"site_name": site_name, "asset_id": asset_id},
    }))["sensors"]
    stats = unwrap(tu.run({
        "name": "iot.sensor_stats",
        "arguments": {"site_name": site_name, "asset_id": asset_id},
    }))["stats"]

    noisiest = sorted(stats, key=lambda s: s.get("stddev") or 0, reverse=True)[:top_n]
    return {
        "asset": f"{detail['asset_id']} ({detail['assettype']}) @ {detail['site_name']}",
        "status": detail["status"],
        "window": f"{extent['start_time']} → {extent['end_time']}",
        "records": extent["total_records"],
        "sensors_installed": len(installed),
        "sensors_measured": len(measured),
        "never_reporting": sorted(set(installed) - set(measured)),
        "most_variable": [
            {"sensor": s["sensor"], "stddev": round(s["stddev"], 2)} for s in noisiest
        ],
    }


tu.register_workflow("asset_snapshot", asset_snapshot)
print("registered:", "asset_snapshot" in tu.list_tools())

In [ ]:
# Invoked exactly like a tool — same entrypoint, same argument shape.
show("workflow: asset_snapshot", tu.run({
    "name": "asset_snapshot",
    "arguments": {"site_name": SITE, "asset_id": ASSET},
}))

Because a workflow takes `tu`, it can call **any** loaded server, not just IoT. Loading
`servers=["iot", "wo", "fmsr"]` lets one workflow join telemetry to work orders and
failure modes, which is the basis of the multi-agent scenarios in the benchmark.

## 8. Clean up

`tu.close()` shuts down the MCP server subprocesses. In a notebook it is easy to forget
and leak them across restarts, so do it explicitly.

In [ ]:
tu.close()
print("closed")

## Where to go next

- [`docs/tool_universe.md`](../docs/tool_universe.md) — the full `ToolUniverse` contract
- [`docs/mcp-servers.md`](../docs/mcp-servers.md) — the other servers (FMSR, TSFM, WO, vibration)
- [`docs/data.md`](../docs/data.md) — loading scenario data instead of the default set
- [`examples/quickstart_tooluniverse.py`](../examples/quickstart_tooluniverse.py) — the same
  three-step contract as a plain script

**Tool reference — everything covered above**

| Tool | Purpose |
|---|---|
| `iot.sites` | list all sites |
| `iot.asset_ids` | asset IDs at a site |
| `iot.assets` | asset records, optional `assettype` filter |
| `iot.asset_detail` | full record for one asset |
| `iot.installed_sensors` | sensors the registry says are fitted |
| `iot.measured_sensors` | sensors actually present in telemetry |
| `iot.find_assets_by_sensors` | reverse lookup: sensors → assets |
| `iot.sensor_coverage` | per-sensor non-null counts and time span |
| `iot.stream_extent` | time range, record count, page-limit warning |
| `iot.latest_reading` | most recent value per sensor |
| `iot.sensor_stats` | min / max / mean / stddev per sensor |
| `iot.history` | raw paginated observations |